# GlitrAI × ComfyUI — Colab Setup

**Assignment 2 — Part A**

Pipeline: Realistic Vision V5.1 (fp16) + dedicated VAE + 4x-UltraSharp upscaler.
Runs img2img at `denoise=0.6` so the reference image is preserved, then upscales 4×.

> ⚠️ **GPU required** — Runtime → Change runtime type → **T4 GPU** before running any cell.

In [ ]:
# Cell 1 — Verify GPU
!nvidia-smi

In [ ]:
# Cell 2 — Clone ComfyUI and install requirements
%cd /content
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!pip install -q -r requirements.txt
print('\n✓ ComfyUI cloned and dependencies installed')

In [ ]:
# Cell 3 — Download Realistic Vision V5.1 fp16 checkpoint (~2.13 GB)
# Using fp16-no-ema variant: same quality, half the download size vs fp32.
# Source: huggingface.co/SG161222/Realistic_Vision_V5.1_noVAE — no login required.
import os

ckpt_dir  = '/content/ComfyUI/models/checkpoints'
os.makedirs(ckpt_dir, exist_ok=True)

ckpt_url  = 'https://huggingface.co/SG161222/Realistic_Vision_V5.1_noVAE/resolve/main/Realistic_Vision_V5.1_fp16-no-ema.safetensors'
ckpt_file = f'{ckpt_dir}/Realistic_Vision_V5.1_fp16-no-ema.safetensors'

print('Downloading Realistic Vision V5.1 fp16 (~2.13 GB) ...')
!wget -q --show-progress -c -O "{ckpt_file}" "{ckpt_url}"
print(f'\n✓ Checkpoint saved')
!ls -lh "{ckpt_dir}"

In [ ]:
# Cell 4 — Download dedicated VAE (~335 MB)
# Realistic_Vision_V5.1_noVAE has no usable baked-in VAE — its model card
# recommends vae-ft-mse-840000-ema-pruned to avoid colour artifacts.
# Source: huggingface.co/stabilityai/sd-vae-ft-mse-original — no login required.
import os

vae_dir  = '/content/ComfyUI/models/vae'
os.makedirs(vae_dir, exist_ok=True)

vae_url  = 'https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors'
vae_file = f'{vae_dir}/vae-ft-mse-840000-ema-pruned.safetensors'

print('Downloading sd-vae-ft-mse (~335 MB) ...')
!wget -q --show-progress -c -O "{vae_file}" "{vae_url}"
print(f'\n✓ VAE saved')
!ls -lh "{vae_dir}"

In [ ]:
# Cell 5 — Download 4x-UltraSharp upscale model (~67 MB)
# Source: huggingface.co/lokCX/4x-Ultrasharp — no login required.
import os

up_dir  = '/content/ComfyUI/models/upscale_models'
os.makedirs(up_dir, exist_ok=True)

up_url  = 'https://huggingface.co/lokCX/4x-Ultrasharp/resolve/main/4x-UltraSharp.pth'
up_file = f'{up_dir}/4x-UltraSharp.pth'

print('Downloading 4x-UltraSharp (~67 MB) ...')
!wget -q --show-progress -c -O "{up_file}" "{up_url}"
print(f'\n✓ Upscale model saved')
!ls -lh "{up_dir}"

In [ ]:
# Cell 6 — Install cloudflared
import platform

arch     = platform.machine()
deb_name = 'cloudflared-linux-arm64.deb' if arch == 'aarch64' else 'cloudflared-linux-amd64.deb'
deb_url  = f'https://github.com/cloudflare/cloudflared/releases/latest/download/{deb_name}'

print(f'Arch: {arch}  →  downloading {deb_name}')
!wget -q -O /tmp/cloudflared.deb "{deb_url}"
!dpkg -i /tmp/cloudflared.deb
!cloudflared --version
print('\n✓ cloudflared installed')

In [ ]:
# Cell 7 — Start ComfyUI (background thread, wait for ready signal)
import subprocess, threading, sys, time

comfyui_ready = threading.Event()

def _run_comfyui():
    proc = subprocess.Popen(
        [sys.executable, 'main.py', '--listen', '0.0.0.0', '--port', '8188'],
        cwd='/content/ComfyUI',
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in proc.stdout:
        print(f'[ComfyUI] {line.rstrip()}')
        if 'To see the GUI go to' in line or 'Starting server' in line:
            comfyui_ready.set()

threading.Thread(target=_run_comfyui, daemon=True).start()

print('Waiting for ComfyUI to start (20-40 s on first run)...')
started = comfyui_ready.wait(timeout=120)
if started:
    time.sleep(3)  # buffer for model scan
    print('\n✓ ComfyUI is running on http://localhost:8188')
else:
    print('\n⚠️  Timed out — check [ComfyUI] lines above for errors')

In [ ]:
# Cell 8 — Start cloudflared tunnel and print the live public URL
import subprocess, threading, re

tunnel_url   = None
tunnel_found = threading.Event()

def _run_tunnel():
    global tunnel_url
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://localhost:8188'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )
    for line in proc.stderr:  # cloudflared writes the URL to stderr
        print(f'[cloudflared] {line.rstrip()}')
        m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
        if m:
            tunnel_url = m.group(0)
            tunnel_found.set()
            break

threading.Thread(target=_run_tunnel, daemon=True).start()

print('Starting cloudflared tunnel...')
found = tunnel_found.wait(timeout=45)

if found:
    print('\n' + '='*64)
    print(f'  COMFYUI PUBLIC URL : {tunnel_url}')
    print('='*64)
    print(f'\n  Open in browser  →  {tunnel_url}')
    print(f'  Set in GlitrAI .env  →  COMFYUI_URL={tunnel_url}')
else:
    print('\n⚠️  Tunnel URL not found — check [cloudflared] lines above')

## ⚠️ Known Limitation — Tunnel URL is Ephemeral

The `trycloudflare.com` URL is **tied to this Colab runtime session**.

| Event | Effect |
|-------|--------|
| Colab disconnect / idle timeout | Tunnel URL dies immediately |
| Runtime recycle (~12 h) | New URL on next run |
| VM restart | Re-run cells 7 and 8 |

**On each restart**: re-run cells 7 and 8, copy the new URL, update `COMFYUI_URL` in `.env`.

This is the expected, documented limitation for free-tier Colab + anonymous cloudflared.
A persistent tunnel would require a Cloudflare account or a cloud GPU service (RunPod, Vast.ai).